# Speech Denoising — Notebook 05: Spectral Loss

Same dilated CNN architecture as notebook 04, but with **spectral loss** instead of MSELoss.

**Key change:** Loss is computed in the frequency domain (STFT magnitude) rather than waveform domain. This reduces phase-related artifacts and better matches human auditory perception.

**Architecture:** Dilated 1D CNN (4 layers) | **Loss:** Spectral (STFT magnitude MSE) with Hann window | **Optimizer:** Adam lr=0.001 | **Epochs:** 4

**Result:** Loss 0.28 → 0.26 (val). Perceptual quality improved vs MSELoss — less metallic sound. Distortion on long files due to training on 1-second chunks (known limitation).

---

## 1. Imports

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm


## 2. Load Preprocessed Data

Train and test chunks saved from notebook 01 (1-second segments, 16000 samples each).

In [ ]:
train_noisy_chunks = np.load('data/train_noisy_chunks.npy')
train_clean_chunks = np.load('data/train_clean_chunks.npy')

## 3. Convert to Tensors

In [ ]:
torch_train_noisy_chunks = torch.from_numpy(train_noisy_chunks)
torch_train_clean_chunks = torch.from_numpy(train_clean_chunks)

## 4. Dataset & DataLoader

`full_dataset` built from train chunks only. Split 80/20 into `train_dataset` and `val_dataset` for training-time validation.

In [ ]:
full_dataset = torch.utils.data.TensorDataset(torch_train_noisy_chunks, torch_train_clean_chunks)

In [ ]:
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32,)
val_dataloader = DataLoader(val_dataset, batch_size=32,)

## 5. Model Architecture

**DenoisingModelDilated** — 4-layer dilated 1D CNN.

- `layer1`: Conv1d(1→16, kernel=3, dilation=1, padding=1)
- `layer2`: Conv1d(16→32, kernel=3, dilation=2, padding=2)
- `layer3`: Conv1d(32→16, kernel=3, dilation=4, padding=4)
- `layer4`: Conv1d(16→1, kernel=3, dilation=8, padding=8)

Dilated convolutions increase receptive field exponentially without extra parameters. Input/output shape: `(batch, 16000)`.

In [ ]:
class DenoisingModelDilated(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, kernel_size=3, dilation=1, padding=1)
        self.layer2 = nn.Conv1d(16, 32, kernel_size=3, dilation=2, padding=2)
        self.layer3 = nn.Conv1d(32, 16, kernel_size=3, dilation=4, padding=4)
        self.layer4 = nn.Conv1d(16, 1, kernel_size=3, dilation=8, padding=8)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [ ]:
model = DenoisingModelDilated()
print(model)

## 6. Spectral Loss

Loss computed on STFT magnitude instead of raw waveform.

`torch.hann_window(1024)` eliminates spectral leakage — without it, FFT bins bleed into neighbours and inflate loss values artificially.

MSE on magnitude = frequency-domain error, closer to how human hearing works.

In [ ]:
def spectral_loss(pred, target):
    window = torch.hann_window(1024)
    stft_target = torch.stft(target, n_fft=1024, return_complex=True, window=window)
    stft_pred = torch.stft(pred, n_fft=1024, return_complex=True, window=window)
    stft_target_magnitude = torch.abs(stft_target)
    stft_pred_magnitude = torch.abs(stft_pred)
    loss = nn.functional.mse_loss(stft_pred_magnitude, stft_target_magnitude)
    return loss

In [ ]:
loss_fn = spectral_loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 7. Training Loop with Validation

4 epochs. Train loss and val loss printed per epoch to monitor overfitting.

In [ ]:
n_epochs = 4

for epoch in tqdm(range(n_epochs)):
    epoch_loss = 0
    for batch in train_dataloader:
        noisy_batch, clean_batch = batch
        optimizer.zero_grad()
        train_pred_clean = model(noisy_batch)
        train_loss = loss_fn(train_pred_clean, clean_batch)
        train_loss.backward()
        optimizer.step()
        epoch_loss += train_loss.item()
    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch}, Avg Loss: {avg_epoch_loss:.4f}")
    
    model.eval()
    with torch.no_grad():
        val_epoch_loss = 0
        for batch in val_dataloader:
            noisy_val_batch, clean_val_batch = batch
            val_pred_clean = model(noisy_val_batch)
            val_loss = loss_fn(val_pred_clean, clean_val_batch)
            val_epoch_loss += val_loss.item()
        avg_val_loss = val_epoch_loss / len(val_dataloader)
        print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    model.train() 

## 8. Save Model Weights

In [ ]:
torch.save(model.state_dict(),'denoising_cnn_spectral_loss_4layers_4epochs.pth')

## 9. Inference

**Known limitation:** model was trained on 1-second chunks (16000 samples). Running on a full-length file (107769 samples) causes distortion — the model generalises poorly beyond its training context length.

Next step: chunked inference with overlap-add to eliminate boundary artifacts.

In [ ]:
test_noisy, _ = lb.load('data/archive/noisy_testset_wav/p232_019.wav', sr=None)
test_noisy_tensor = torch.from_numpy(test_noisy).unsqueeze(0)
pred_clean = model(test_noisy_tensor).detach().numpy().squeeze(0)

In [ ]:
Audio(test_noisy, rate=16000)

In [ ]:
Audio(pred_clean, rate=16000)